# 06 - Vision Transformer Architecture for Packet Classification

This notebook implements and tests the Vision Transformer (ViT) architecture specifically optimized for network packet malware detection. We'll build upon our patch embedding and position embedding modules to create a complete ViT model.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

# For architecture visualization
from torchinfo import summary
import time

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')

# Configure notebook display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Setup project paths
notebook_path = Path().resolve()
project_root = notebook_path.parent

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our modules
from src.models.patch_embed import create_patch_embedding
from src.models.position_embed import create_position_embedding
from src.data.packet_to_image import PacketImageEncoder

print(f"Project root: {project_root}")

## 1. Core ViT Components

### 1.1 Multi-Head Self-Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Self-Attention module for Vision Transformer.
    Includes optimizations for packet data processing.
    """
    
    def __init__(self,
                 dim: int,
                 num_heads: int = 8,
                 qkv_bias: bool = True,
                 attn_drop: float = 0.,
                 proj_drop: float = 0.):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} must be divisible by num_heads {num_heads}"
        
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        # Single matrix for Q, K, V projections
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        
    def forward(self, x: torch.Tensor, return_attention: bool = False) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (B, N, C)
            return_attention: Whether to return attention weights
            
        Returns:
            Output tensor of shape (B, N, C)
            Optional attention weights of shape (B, num_heads, N, N)
        """
        B, N, C = x.shape
        
        # Generate Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)  # Each is (B, num_heads, N, head_dim)
        
        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        
        if return_attention:
            return x, attn
        return x


class PacketAwareAttention(MultiHeadAttention):
    """
    Attention module with packet-specific biases.
    Incorporates knowledge about packet structure into attention computation.
    """
    
    def __init__(self,
                 dim: int,
                 num_heads: int = 8,
                 qkv_bias: bool = True,
                 attn_drop: float = 0.,
                 proj_drop: float = 0.,
                 use_packet_bias: bool = True):
        super().__init__(dim, num_heads, qkv_bias, attn_drop, proj_drop)
        self.use_packet_bias = use_packet_bias
        
        if use_packet_bias:
            # Learnable bias for header-header, header-payload, payload-payload attention
            self.packet_bias = nn.Parameter(torch.zeros(3, num_heads, 1, 1))
            nn.init.normal_(self.packet_bias, std=0.02)
    
    def forward(self, x: torch.Tensor, 
                patch_types: Optional[torch.Tensor] = None,
                return_attention: bool = False) -> torch.Tensor:
        """
        Args:
            x: Input tensor (B, N, C)
            patch_types: Tensor indicating patch type (0=header, 1=payload) (B, N)
            return_attention: Whether to return attention weights
        """
        B, N, C = x.shape
        
        # Standard attention computation
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add packet-aware bias if enabled
        if self.use_packet_bias and patch_types is not None:
            # Create bias matrix based on patch types
            bias = torch.zeros(B, self.num_heads, N, N, device=x.device)
            
            for b in range(B):
                header_mask = patch_types[b] == 0
                payload_mask = patch_types[b] == 1
                
                # Header-header attention bias
                bias[b, :, header_mask, :][:, :, header_mask] += self.packet_bias[0]
                
                # Header-payload attention bias  
                bias[b, :, header_mask, :][:, :, payload_mask] += self.packet_bias[1]
                bias[b, :, payload_mask, :][:, :, header_mask] += self.packet_bias[1]
                
                # Payload-payload attention bias
                bias[b, :, payload_mask, :][:, :, payload_mask] += self.packet_bias[2]
            
            attn = attn + bias
        
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        
        if return_attention:
            return x, attn
        return x

### 1.2 MLP (Feed-Forward Network)


In [ ]:
class MLP(nn.Module):
    """
    MLP module for Vision Transformer.
    Two-layer feed-forward network with GELU activation.
    """
    
    def __init__(self,
                 in_features: int,
                 hidden_features: Optional[int] = None,
                 out_features: Optional[int] = None,
                 act_layer: nn.Module = nn.GELU,
                 drop: float = 0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


class GatedMLP(nn.Module):
    """
    Gated MLP variant that can learn to filter features.
    Useful for packet data where some features might be noise.
    """
    
    def __init__(self,
                 in_features: int,
                 hidden_features: Optional[int] = None,
                 out_features: Optional[int] = None,
                 act_layer: nn.Module = nn.GELU,
                 drop: float = 0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        
        # Main path
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.drop1 = nn.Dropout(drop)
        
        # Gating path
        self.fc_gate = nn.Linear(in_features, hidden_features)
        self.gate_act = nn.Sigmoid()
        
        # Output projection
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Main path
        feat = self.fc1(x)
        feat = self.act(feat)
        
        # Gating
        gate = self.fc_gate(x)
        gate = self.gate_act(gate)
        
        # Apply gate
        x = feat * gate
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        
        return x

### 1.3 Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    """
    Transformer block for Vision Transformer.
    Consists of Multi-Head Attention and MLP with residual connections.
    """
    
    def __init__(self,
                 dim: int,
                 num_heads: int,
                 mlp_ratio: float = 4.,
                 qkv_bias: bool = True,
                 drop: float = 0.,
                 attn_drop: float = 0.,
                 act_layer: nn.Module = nn.GELU,
                 norm_layer: nn.Module = nn.LayerNorm,
                 attention_type: str = 'standard',
                 mlp_type: str = 'standard'):
        super().__init__()
        
        # Normalization layers
        self.norm1 = norm_layer(dim)
        self.norm2 = norm_layer(dim)
        
        # Attention module
        if attention_type == 'standard':
            self.attn = MultiHeadAttention(
                dim, num_heads=num_heads, qkv_bias=qkv_bias,
                attn_drop=attn_drop, proj_drop=drop
            )
        elif attention_type == 'packet_aware':
            self.attn = PacketAwareAttention(
                dim, num_heads=num_heads, qkv_bias=qkv_bias,
                attn_drop=attn_drop, proj_drop=drop
            )
        else:
            raise ValueError(f"Unknown attention type: {attention_type}")
        
        # MLP module
        mlp_hidden_dim = int(dim * mlp_ratio)
        if mlp_type == 'standard':
            self.mlp = MLP(
                in_features=dim, hidden_features=mlp_hidden_dim,
                act_layer=act_layer, drop=drop
            )
        elif mlp_type == 'gated':
            self.mlp = GatedMLP(
                in_features=dim, hidden_features=mlp_hidden_dim,
                act_layer=act_layer, drop=drop
            )
        else:
            raise ValueError(f"Unknown MLP type: {mlp_type}")
        
        # Stochastic depth (drop path)
        self.drop_path = nn.Identity()  # Can be replaced with DropPath
        
    def forward(self, x: torch.Tensor, return_attention: bool = False) -> torch.Tensor:
        """
        Args:
            x: Input tensor (B, N, C)
            return_attention: Whether to return attention weights
        """
        # Self-attention with residual
        if return_attention:
            attn_out, attn_weights = self.attn(self.norm1(x), return_attention=True)
            x = x + self.drop_path(attn_out)
        else:
            x = x + self.drop_path(self.attn(self.norm1(x)))
            attn_weights = None
        
        # MLP with residual
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        
        if return_attention:
            return x, attn_weights
        return x


class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample."""
    
    def __init__(self, drop_prob: float = 0.):
        super().__init__()
        self.drop_prob = drop_prob
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0. or not self.training:
            return x
            
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # binarize
        output = x.div(keep_prob) * random_tensor
        return output

## 2. Complete Vision Transformer Architecture

In [ ]:
class VisionTransformerForPackets(nn.Module):
    """
    Vision Transformer optimized for network packet classification.
    
    Includes packet-specific optimizations:
    - Multiple patch embedding strategies
    - Byte-order aware position embeddings
    - Packet-aware attention mechanisms
    - Configurable architectures (tiny, small, base)
    """
    
    def __init__(self,
                 image_size: int = 224,
                 patch_size: int = 16,
                 in_channels: int = 1,
                 num_classes: int = 6,  # Benign + 5 attack types
                 embed_dim: int = 768,
                 depth: int = 12,
                 num_heads: int = 12,
                 mlp_ratio: float = 4.,
                 qkv_bias: bool = True,
                 representation_size: Optional[int] = None,
                 drop_rate: float = 0.,
                 attn_drop_rate: float = 0.,
                 drop_path_rate: float = 0.,
                 embed_layer: str = 'standard',
                 pos_embed_type: str = 'learnable',
                 attention_type: str = 'standard',
                 mlp_type: str = 'standard',
                 norm_layer: nn.Module = nn.LayerNorm,
                 act_layer: nn.Module = nn.GELU,
                 cls_token: bool = True,
                 **kwargs):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.cls_token = cls_token
        
        # Patch embedding
        self.patch_embed = create_patch_embedding(
            image_size, patch_size, in_channels, embed_dim, embed_layer, **kwargs
        )
        
        # Calculate number of patches
        if hasattr(self.patch_embed, 'num_patches'):
            num_patches = self.patch_embed.num_patches
        else:
            num_patches = (image_size // patch_size) ** 2
            
        # Class token
        if cls_token:
            self.cls_token_param = nn.Parameter(torch.zeros(1, 1, embed_dim))
            nn.init.trunc_normal_(self.cls_token_param, std=0.02)
        
        # Position embedding
        self.pos_embed = create_position_embedding(
            num_patches, embed_dim, pos_embed_type, 
            cls_token=cls_token, patch_size=patch_size, 
            image_size=image_size, **kwargs
        )
        
        # Dropout after position embedding
        self.pos_drop = nn.Dropout(p=drop_rate)
        
        # Stochastic depth decay rule
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                act_layer=act_layer,
                norm_layer=norm_layer,
                attention_type=attention_type,
                mlp_type=mlp_type
            )
            for i in range(depth)
        ])
        
        # Add drop path to blocks
        for i, block in enumerate(self.blocks):
            block.drop_path = DropPath(dpr[i]) if dpr[i] > 0. else nn.Identity()
        
        # Final norm
        self.norm = norm_layer(embed_dim)
        
        # Representation layer (optional)
        if representation_size:
            self.num_features = representation_size
            self.pre_logits = nn.Sequential(
                nn.Linear(embed_dim, representation_size),
                nn.Tanh()
            )
        else:
            self.pre_logits = nn.Identity()
            
        # Classification head
        self.head = nn.Linear(self.num_features, num_classes)
        
        # Initialize weights
        self.apply(self._init_weights)
        
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
            
    def forward_features(self, x: torch.Tensor, 
                        return_all_tokens: bool = False,
                        return_attention: bool = False) -> torch.Tensor:
        """
        Forward pass through feature extraction.
        
        Args:
            x: Input images (B, C, H, W)
            return_all_tokens: Return all tokens instead of just CLS
            return_attention: Return attention maps from all layers
        """
        # Patch embedding
        x = self.patch_embed(x)  # (B, num_patches, embed_dim)
        
        # Add CLS token
        if self.cls_token:
            cls_tokens = self.cls_token_param.expand(x.shape[0], -1, -1)
            x = torch.cat((cls_tokens, x), dim=1)
        
        # Add position embedding
        x = self.pos_embed(x)
        x = self.pos_drop(x)
        
        # Apply transformer blocks
        attention_maps = []
        for block in self.blocks:
            if return_attention:
                x, attn = block(x, return_attention=True)
                attention_maps.append(attn)
            else:
                x = block(x)
        
        # Final normalization
        x = self.norm(x)
        
        if return_all_tokens:
            return x, attention_maps if return_attention else x
        else:
            # Return CLS token or global average pool
            if self.cls_token:
                return x[:, 0], attention_maps if return_attention else x[:, 0]
            else:
                return x.mean(dim=1), attention_maps if return_attention else x.mean(dim=1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Full forward pass.
        
        Args:
            x: Input images (B, C, H, W)
            
        Returns:
            Class logits (B, num_classes)
        """
        x = self.forward_features(x)
        x = self.pre_logits(x)
        x = self.head(x)
        return x
    
    def get_attention_maps(self, x: torch.Tensor) -> List[torch.Tensor]:
        """Get attention maps from all layers."""
        _, attention_maps = self.forward_features(x, return_attention=True)
        return attention_maps
    
    def get_intermediate_features(self, x: torch.Tensor, 
                                layer_indices: List[int]) -> Dict[int, torch.Tensor]:
        """Extract features from intermediate layers."""
        features = {}
        
        # Patch embedding
        x = self.patch_embed(x)
        
        # Add CLS token
        if self.cls_token:
            cls_tokens = self.cls_token_param.expand(x.shape[0], -1, -1)
            x = torch.cat((cls_tokens, x), dim=1)
        
        # Add position embedding
        x = self.pos_embed(x)
        x = self.pos_drop(x)
        
        # Apply transformer blocks
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i in layer_indices:
                features[i] = x.clone()
        
        return features

## 3. Model Configurations


In [ ]:
def vit_tiny_patch16_64(**kwargs):
    """ViT-Tiny for 64x64 packet images."""
    model = VisionTransformerForPackets(
        image_size=64,
        patch_size=16,
        embed_dim=192,
        depth=12,
        num_heads=3,
        mlp_ratio=4,
        **kwargs
    )
    return model

def vit_small_patch16_128(**kwargs):
    """ViT-Small for 128x128 packet images."""
    model = VisionTransformerForPackets(
        image_size=128,
        patch_size=16,
        embed_dim=384,
        depth=12,
        num_heads=6,
        mlp_ratio=4,
        **kwargs
    )
    return model

def vit_base_patch16_224(**kwargs):
    """ViT-Base for 224x224 packet images."""
    model = VisionTransformerForPackets(
        image_size=224,
        patch_size=16,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4,
        **kwargs
    )
    return model

def vit_packet_custom(**kwargs):
    """Custom ViT configuration optimized for packet data."""
    model = VisionTransformerForPackets(
        image_size=128,
        patch_size=16,
        embed_dim=512,
        depth=8,
        num_heads=8,
        mlp_ratio=3,
        embed_layer='hybrid',
        pos_embed_type='byte_aware',
        attention_type='packet_aware',
        mlp_type='gated',
        drop_rate=0.1,
        attn_drop_rate=0.1,
        drop_path_rate=0.1,
        **kwargs
    )
    return model

# Test model creation and summary
def test_model_configs():
    """Test different model configurations."""
    configs = {
        'ViT-Tiny (64x64)': vit_tiny_patch16_64,
        'ViT-Small (128x128)': vit_small_patch16_128,
        'ViT-Base (224x224)': vit_base_patch16_224,
        'ViT-Packet-Custom': vit_packet_custom
    }
    
    for name, model_fn in configs.items():
        print(f"\n{'='*60}")
        print(f"{name}")
        print('='*60)
        
        model = model_fn(num_classes=6)
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Model size: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")
        
        # Test forward pass
        if '64' in name:
            x = torch.randn(2, 1, 64, 64)
        elif '128' in name or 'Custom' in name:
            x = torch.randn(2, 1, 128, 128)
        else:
            x = torch.randn(2, 1, 224, 224)
            
        with torch.no_grad():
            output = model(x)
            print(f"Output shape: {output.shape}")

test_model_configs()

## 4. Attention Visualization

In [ ]:
def visualize_attention_maps(model: nn.Module, 
                           image: torch.Tensor,
                           layer_idx: int = -1,
                           head_idx: Optional[int] = None):
    """
    Visualize attention maps from a specific layer and head.
    
    Args:
        model: Vision Transformer model
        image: Input image tensor (1, C, H, W)
        layer_idx: Which transformer layer to visualize (-1 for last)
        head_idx: Which attention head to visualize (None for average)
    """
    model.eval()
    
    with torch.no_grad():
        # Get attention maps
        attention_maps = model.get_attention_maps(image)
        
        # Select layer
        if layer_idx == -1:
            layer_idx = len(attention_maps) - 1
        attn = attention_maps[layer_idx]  # (B, num_heads, N, N)
        
        # Select head
        if head_idx is not None:
            attn = attn[:, head_idx, :, :]  # (B, N, N)
        else:
            attn = attn.mean(dim=1)  # Average over heads
        
        # Focus on CLS token attention (if using CLS token)
        if model.cls_token:
            attn = attn[0, 0, 1:]  # Attention from CLS to patches
        else:
            attn = attn[0].mean(dim=0)  # Average attention
        
        # Reshape to 2D
        num_patches = int(np.sqrt(len(attn)))
        attn = attn.reshape(num_patches, num_patches)
        
        # Visualize
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        
        # Original image
        ax = axes[0]
        img_np = image[0, 0].cpu().numpy()
        ax.imshow(img_np, cmap='gray')
        ax.set_title('Original Packet Image')
        ax.axis('off')
        
        # Attention map
        ax = axes[1]
        im = ax.imshow(attn.cpu().numpy(), cmap='hot', interpolation='nearest')
        ax.set_title(f'Attention Map (Layer {layer_idx})')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
        
        plt.tight_layout()
        plt.show()

# Test attention visualization
def test_attention_visualization():
    """Test attention visualization with sample data."""
    # Create model
    model = vit_tiny_patch16_64(num_classes=6)
    model.eval()
    
    # Create sample image (simulate packet data)
    image = torch.zeros(1, 1, 64, 64)
    # Add some patterns
    image[0, 0, :20, :20] = 0.8  # Header region
    image[0, 0, 30:40, 30:40] = 0.6  # Some payload pattern
    image[0, 0, 50:, 50:] = 0.3  # Another pattern
    
    # Add noise
    noise = torch.randn_like(image) * 0.1
    image = torch.clamp(image + noise, 0, 1)
    
    # Visualize attention for different layers
    for layer_idx in [0, 5, -1]:
        print(f"\nVisualizing attention for layer {layer_idx}")
        visualize_attention_maps(model, image, layer_idx=layer_idx)

test_attention_visualization()

## 5. Performance Analysis

In [ ]:
# Performance benchmarking
def benchmark_model_performance(model: nn.Module, 
                              input_size: Tuple[int, int],
                              batch_sizes: List[int] = [1, 4, 16, 32],
                              num_runs: int = 100):
    """Benchmark model inference performance."""
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    results = {}
    
    for batch_size in batch_sizes:
        # Create dummy input
        x = torch.randn(batch_size, 1, *input_size).to(device)
        
        # Warmup
        for _ in range(10):
            with torch.no_grad():
                _ = model(x)
        
        # Time inference
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            
        start_time = time.time()
        
        for _ in range(num_runs):
            with torch.no_grad():
                _ = model(x)
                
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            
        end_time = time.time()
        
        # Calculate metrics
        total_time = end_time - start_time
        avg_time = total_time / num_runs
        throughput = batch_size / avg_time
        
        results[batch_size] = {
            'avg_time_ms': avg_time * 1000,
            'throughput_samples_per_sec': throughput,
            'total_time': total_time
        }
    
    return results

# Benchmark different models
def compare_model_performance():
    """Compare performance of different model variants."""
    models = {
        'ViT-Tiny': (vit_tiny_patch16_64(), (64, 64)),
        'ViT-Small': (vit_small_patch16_128(), (128, 128)),
        'ViT-Custom': (vit_packet_custom(), (128, 128))
    }
    
    all_results = {}
    
    for name, (model, input_size) in models.items():
        print(f"\nBenchmarking {name}...")
        results = benchmark_model_performance(model, input_size, num_runs=50)
        all_results[name] = results
    
    # Create comparison table
    comparison_data = []
    for model_name, results in all_results.items():
        for batch_size, metrics in results.items():
            comparison_data.append({
                'Model': model_name,
                'Batch Size': batch_size,
                'Avg Time (ms)': f"{metrics['avg_time_ms']:.2f}",
                'Throughput (samples/s)': f"{metrics['throughput_samples_per_sec']:.1f}"
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n\nPerformance Comparison:")
    print(comparison_df.to_string(index=False))
    
    # Visualize results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Inference time
    for model_name in all_results:
        batch_sizes = list(all_results[model_name].keys())
        avg_times = [all_results[model_name][bs]['avg_time_ms'] for bs in batch_sizes]
        ax1.plot(batch_sizes, avg_times, marker='o', label=model_name)
    
    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('Average Inference Time (ms)')
    ax1.set_title('Inference Time vs Batch Size')
    ax1.legend()
    ax1.grid(True)
    
    # Throughput
    for model_name in all_results:
        batch_sizes = list(all_results[model_name].keys())
        throughputs = [all_results[model_name][bs]['throughput_samples_per_sec'] for bs in batch_sizes]
        ax2.plot(batch_sizes, throughputs, marker='o', label=model_name)
    
    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('Throughput (samples/sec)')
    ax2.set_title('Throughput vs Batch Size')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Run performance comparison
compare_model_performance()